# Local data and pipeline testing

## Making VS Code find Java

In [ ]:
!/usr/libexec/java_home -v 17

In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.20.1/libexec/openjdk.jdk/Contents/Home"

In [ ]:
import os
print(os.environ.get("JAVA_HOME"))
print(os.environ.get("PATH"))

## Data size (for Athena scan + AWS storage pricing)

Content length reading script: `check_data_size.sh`.

In [ ]:
import numpy as np

In [ ]:
with open("data_size.txt", "r") as f:
    NBytes = np.sum([
        int(line.split(': ')[1].rstrip('\n'))
        for line in f.readlines()
    ])
    NGB = NBytes / 1024 ** 3
    print(NGB)

Byes scanned per query:

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

query = """
SELECT *
FROM read_parquet('data/yellow_tripdata_2024-*.parquet');
"""
query_analysis = "\n".join(["EXPLAIN ANALYZE", query])


print("".join(con.execute(query_analysis).fetchone()))

df = con.execute(query).df()
print(df.count())



## Basic data query, dataviz

In [ ]:
# using cloudfront data link instead of aws for local testing

df = con.execute("""
    SELECT *
    FROM read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet')
    LIMIT 10000
""").df()
print(df.head())

## Data wrangling in distributed context test

Data download: `aws-nyc-taxi/data_download.sh`

### Read downloaded data

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("nyc-taxi-local")
    .master("local[*]")          # uses all local cores, no cluster
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

df = spark.read.parquet("data/yellow_tripdata_2024-*.parquet")
df.printSchema()
df.show(5)

### Data aggregation

In [ ]:
from pyspark.sql import functions as F

hourly = (
    df.withColumn("pickup_hour", F.date_trunc("hour", "tpep_pickup_datetime"))
      .groupBy("PULocationID", "pickup_hour")
      .agg(
          F.count("*").alias("trip_count"),
          F.avg("fare_amount").alias("avg_fare"),
          F.std("fare_amount").alias("stdev_fare"),
          F.avg("tip_amount").alias("avg_tip"),
          F.std("tip_amount").alias("stdev_tip"),
      ).withColumn("pickup_hour_only", F.hour("pickup_hour"))
)
hourly.orderBy(F.desc("trip_count")).show(10)

In [ ]:
(
    hourly
      .filter("avg_fare <= 100") # filter out outliers
      .filter("avg_tip<=25")
      .plot.scatter(x="avg_fare", y="avg_tip", color="pickup_hour_only")
      .show()
)

### Window

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("PULocationID").orderBy("pickup_hour")
hourly_with_lag = (
    hourly.withColumn(
        "prev_hour_count", F.lag("trip_count", 1).over(w)
    )
    .withColumn(
        "2prev_hour_count", F.lag("trip_count", 2).over(w)
    ).withColumn(
        "3prev_hour_count", F.lag("trip_count", 3).over(w)
    )
)
hourly_with_lag.show(10)

In [ ]:
hourly_with_lag.plot.scatter(
    x="prev_hour_count",
    y="trip_count",
)

In [ ]:
hourly_with_lag.plot.scatter(
    x="2prev_hour_count",
    y="trip_count",
)

In [ ]:
hourly_with_lag.plot.scatter(
    x="3prev_hour_count",
    y="trip_count",
)


In [ ]:
# Pearson correlations

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# 0. drop NA for VectorAssembler
hourly_with_lag=hourly_with_lag.dropna(
    subset=[
        "trip_count", 
        "prev_hour_count", 
        "2prev_hour_count",
        "3prev_hour_count",
    ]
)

# 1. Prepare your data into a feature vector
assembler = VectorAssembler(inputCols=["trip_count", "prev_hour_count"], outputCol="features")
vector_df = assembler.transform(hourly_with_lag)


# 2. Compute the Pearson correlation matrix
matrix = Correlation.corr(vector_df, "features").head()[0]

# # 3. Extract the correlation value and square it to get R2
r_value = float(matrix[0, 1])
r2_value = r_value ** 2

print(f"count vs lag-1 count R2 Value: {r2_value}")

In [ ]:
# Pearson correlations

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# 1. Prepare your data into a feature vector
assembler = VectorAssembler(inputCols=["trip_count", "2prev_hour_count"], outputCol="features")
vector_df = assembler.transform(hourly_with_lag)


# 2. Compute the Pearson correlation matrix
matrix = Correlation.corr(vector_df, "features").head()[0]

# # 3. Extract the correlation value and square it to get R2
r_value = float(matrix[0, 1])
r2_value = r_value ** 2

print(f"count vs lag-2 count R2 Value: {r2_value}")

In [ ]:
# Pearson correlations

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# 1. Prepare your data into a feature vector
assembler = VectorAssembler(inputCols=["trip_count", "3prev_hour_count"], outputCol="features")
vector_df = assembler.transform(hourly_with_lag)


# 2. Compute the Pearson correlation matrix
matrix = Correlation.corr(vector_df, "features").head()[0]

# # 3. Extract the correlation value and square it to get R2
r_value = float(matrix[0, 1])
r2_value = r_value ** 2

print(f"count vs lag-3 count R2 Value: {r2_value}")

### Revenue as fn of previous trip count

In [ ]:
hourly_with_lag = (
    hourly_with_lag
    .withColumn("total_revenue", F.col("avg_fare") * F.col("trip_count"))
)

hourly_with_lag.plot.scatter(x="prev_hour_count", y="total_revenue")

## ML

In [ ]:
# count rows to estimate computational complexity:

hourly_with_lag.count()


### Ridge regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.kernel_ridge import KernelRidge

In [ ]:
feature_cols = ["prev_hour_count", "2prev_hour_count", "3prev_hour_count"] 
label_col = "trip_count"

kr_df = hourly_with_lag.select(feature_cols + [label_col]).dropna().toPandas()

kr_df = kr_df.sample(5000) # for local testing

X = kr_df[feature_cols].values
y = kr_df[label_col].values
print(len(y))

In [ ]:
kr_model = KernelRidge(kernel="rbf", alpha=1.0)
kr_model.fit(X, y)

In [ ]:
import numpy as np

kr_model.predict(np.array([[5,3,2], [7,4,9]]))

In [ ]:
kr_test_df = hourly_with_lag.select(feature_cols + [label_col]).dropna().toPandas()

kr_test_df = kr_test_df.sample(1000)

In [ ]:
kr_test_df[feature_cols].to_numpy()

In [ ]:
from sklearn.metrics import mean_squared_error

kr_test_df['prediction'] = (
    kr_model.predict(kr_test_df[feature_cols].to_numpy())
)

mse = mean_squared_error(kr_test_df['trip_count'], kr_test_df['prediction'])

print(f"Kernel Ridge mse: {mse}")

### XGBoost

In [ ]:
from xgboost.spark import SparkXGBRegressor
xgb_regressor = SparkXGBRegressor(
  features_col=["prev_hour_count","2prev_hour_count", "3prev_hour_count"],
  label_col="trip_count",
  num_workers=8,
)

In [ ]:
# sample small training and testing datasets
xgb_train_df = hourly_with_lag.sample(0.04)
xgb_test_df_with_gt = hourly_with_lag.sample(0.01).withColumn("ground_truth", F.col("trip_count"))

xgb_test_df = xgb_test_df_with_gt.select(
    [col for col in xgb_test_df_with_gt.columns if col != "trip_count"]
)
print(f"{xgb_train_df.count()}, {xgb_test_df.count()}")

In [ ]:
xgb_test_df.columns

In [ ]:
model = xgb_regressor.fit(xgb_train_df);

In [ ]:
prediction = model.transform(xgb_test_df)

In [ ]:
prediction.show()

In [ ]:
mse_result = prediction.agg(F.mean((F.col("ground_truth") - F.col("prediction")) ** 2).alias("mse")).collect()
mse = mse_result[0]["mse"]

print(f"XGBoost mse: {mse}")

## Benchmarking train time

SageMaker pricing depends on wall-time, not total CPU time. Test different number of jobs (thinking 1 job per CPU) and different number of rows to get a table:

| runtime (s) | job-count | instance price |
|-------------|--------------|----------------|
| ... | ... | ... |
| ... | ... | ... |

For more control on the output, I fix the number of estimators (trees) and max tree depth.


In [ ]:
import time

SAMPLE_FRACTIONS = [0.01, 0.02, 0.04, 0.08, 0.16, 0.24, 0.32, 0.40]

train_df_dict = {
    sample_frac: (
        hourly_with_lag
            .select([*feature_cols, label_col])
            .sample(sample_frac)
            .toPandas()
    )
    for sample_frac in SAMPLE_FRACTIONS
}
X_dict = {
    sample_frac: train_df_dict[sample_frac][feature_cols].to_numpy()
    for sample_frac in SAMPLE_FRACTIONS
}
y_dict = {
    sample_frac: train_df_dict[sample_frac][[label_col]].to_numpy()
    for sample_frac in SAMPLE_FRACTIONS
}

print(X_dict[SAMPLE_FRACTIONS[0]].shape)
print(y_dict[SAMPLE_FRACTIONS[0]].shape)

In [ ]:
import xgboost as xgb
import pandas as pd
# data structure 'n_rows', 'n_jobs', 'wall_runtime', 'sage_instance', 'instance_price'
benchmark_data = []

# let's use 8GiB instances that are close to the 8GB of ram 
# available locally
instances = {
    2: 'sc.t3.large',
    4: 'sc.t3.xlarge',
    8: 'sc.t3.2xlarge',
}
prices = { # USD
    2: 0.1,
    4: 0.2,
    8: 0.399,
}

for n_jobs in [2, 4]:
    for sample_frac in SAMPLE_FRACTIONS:
        X = X_dict[sample_frac]
        y = y_dict[sample_frac]
        start = time.time()
        model = xgb.XGBRegressor(n_estimators=200, max_depth=6, n_jobs=n_jobs)
        model.fit(X, y)  
        runtime = time.time() - start
        benchmark_data.append([
            y.shape[0], n_jobs, runtime, instances[n_jobs], prices[n_jobs],
        ])
        print(
            f"n_jobs={n_jobs}, price={prices[n_jobs]}, "
            f"runtime={runtime:.1f}s"
        )

benchmark_df = pd.DataFrame(
    benchmark_data,
    columns=[
        'n_rows', 'n_jobs', 'wall_runtime', 
        'sage_instance', 'instance_price'
    ]
)

In [ ]:
benchmark_df['runtime_price'] = benchmark_df.apply(
    axis=1,
    func = lambda row: row.instance_price * row.wall_runtime
)

In [ ]:
benchmark_df.sort_values(by='n_rows').head()

In [ ]:
benchmark_df.plot(
    x = 'n_rows',
    y = 'runtime_price',
    c = 'n_jobs',
    cmap = 'viridis',
    kind='scatter'
)

In [ ]:
# linear regression

from sklearn.linear_model import LinearRegression

X_nj2 = np.array([
    [x]
    for x in benchmark_df[benchmark_df['n_jobs']==2].n_rows.to_numpy()
])
y_nj2 = benchmark_df[benchmark_df['n_jobs']==2].runtime_price.to_numpy()

X_nj4 = np.array([
    [x]
    for x in benchmark_df[benchmark_df['n_jobs']==4].n_rows.to_numpy()
])
y_nj4 = benchmark_df[benchmark_df['n_jobs']==4].runtime_price.to_numpy()

reg_nj2 = LinearRegression().fit(X_nj2, y_nj2)
reg_nj4 = LinearRegression().fit(X_nj4, y_nj4)

In [ ]:
tot_rows = hourly_with_lag.count()
tot_rows

In [ ]:
print(f"2-job full-scale price estimate: ${reg_nj2.predict([[tot_rows]])[0]:.2f}")
print(f"4-job full-scale price estimate: ${reg_nj4.predict([[tot_rows]])[0]:.2f}")